In [3]:
# coppelia library installation
! pip install coppeliasim-zmqremoteapi-client pyzmq

Defaulting to user installation because normal site-packages is not writeable


In [1]:
from time import sleep
from coppeliasim_zmqremoteapi_client import RemoteAPIClient
import cv2
import numpy as np
from matplotlib.patches import Circle
import matplotlib.pyplot as plt 

In [5]:
client = RemoteAPIClient()
sim = client.getObject('sim')
cap = cv2.VideoCapture(0)
sim.startSimulation() #start simulation
target = sim.getObject('/target') # get target object for control the objet in coppelia
HisPosi = np.random.randint(0,1,(2,3)) #history of the green ball's position
HisPosi[:,2]=0.5
# initialization 
# bounding for green color
lowerband = np.array([35, 100, 10])
upperband = np.array([85,255, 255])

ret,frame = cap.read()
frame = cv2.cvtColor(frame,cv2.COLOR_BGR2HSV)
mask = cv2.inRange(frame,lowerband,upperband)
contour,_ = cv2.findContours(mask,mode=cv2.RETR_TREE,method=cv2.CHAIN_APPROX_SIMPLE)
if contour:
    max_contour = max(contour,key=cv2.contourArea)
    if cv2.contourArea(max_contour)>500:
        x,y,w,h = cv2.boundingRect(max_contour)
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
        HisPosi[0,:1] = x+w,y+h
        sim.setObjectPosition(target,HisPosi[0,:],sim.handle_parent)
sleep(2)
try:
    while sim.getSimulationTime()<30:
        HisPosi[1,:]=HisPosi[0,:]
        ret,frame = cap.read()
        lowerband = np.array([35, 100, 10])
        upperband = np.array([85,255, 255])
        frame = cv2.cvtColor(frame,cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(frame,lowerband,upperband)
        contour,_ = cv2.findContours(mask,mode=cv2.RETR_TREE,method=cv2.CHAIN_APPROX_SIMPLE)
        if contour:
            max_contour = max(contour,key=cv2.contourArea)
            if cv2.contourArea(max_contour)>500:
                x,y,w,h = cv2.boundingRect(max_contour)
                cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
                HisPosi[0,:1] = x+w,y+h
                dist = (HisPosi[1,:]-HisPosi[0,:])*0.01
                print(dist)
        sleep(2)
        # target_position = np.array(sim.getObjectPosition(target))
        # target_position[0:1]=target_position[0,1]+dist
        print(HisPosi)
        # sim.setObjectPosition(target,[0.5,0.5,0.5],sim.handle_parent)
        new_frame = cv2.cvtColor(frame,cv2.COLOR_HSV2BGR)
        cv2.imshow('new_img',new_frame)
        if cv2.waitKey(1)& 0xFF == ord('q'):
            break
        
    sim.stopSimulation()
    cap.release()
    cv2.destroyAllWindows()
    
except:
    print("Error")
    sim.stopSimulation()
    cap.release()
    cv2.destroyAllWindows()



[[0 0]
 [0 0]]

[[0 0]
 [0 0]]

[[387 207]
 [  0   0]]

[[387 207]
 [407 218]]

[[391 203]
 [407 218]]

[[391 203]
 [407 218]]

[[391 203]
 [408 228]]


In [6]:
# sensor = sim.getObject('/proximitySensor')
# sensor2Handle = sim.getObject('/PassiveVisionSensor')
sim = client.getObject('sim')
sim.startSimulation()


while sim.getSimulationTime() < 40:
    sleep(2)
    sim.setObjectPosition(target,[1,1,0.5],sim.handle_parent)
    sleep(3)
    sim.setObjectPosition(target,[1,-0.5,1],sim.handle_parent)
    sleep(2)
    sim.setObjectPosition(target,[-0.5,-0.5,1],sim.handle_parent)
    sim.step()
    sleep(0.01)
sim.stopSimulation()

NameError: name 'target' is not defined